# Stock Market Risk & Correlation Analyzer
### Quantitative Financial Econometrics, Statistical Battery & Portfolio Optimization

This notebook provides a fully reproducible, rigorous quantitative statistical analysis of the 10 approved National Stock Exchange of India (NSE) equities:
1. **Descriptive Statistics & Dispersion**: Daily Mean, Median, 5% Trimmed Mean, Bessel's Correction Sample Variance ($ddof=1$), Annualized Volatility ($\sqrt{252}$), Skewness, Excess Kurtosis.
2. **Parametric & Non-Parametric Confidence Intervals (95%)**: Student's $t$ for Mean Returns, Chi-Square ($\chi^2$) for Volatility, Fisher $z$-transformation for Pearson correlation.
3. **Econometric Assumption Checks**: Normality (Jarque-Bera, Shapiro-Wilk), Stationarity (Augmented Dickey-Fuller unit root), Serial Independence (Durbin-Watson, Ljung-Box), Autoregressive Conditional Heteroskedasticity (ARCH volatility clustering).
4. **Hypothesis Testing & Standardized Effect Sizes**: Student's $t$-test on returns ($H_0: \mu = 0$), Cohen's $d$, Annualized Sharpe Ratios ($R_f=6.5\%$), Pairwise correlation $t$-tests ($H_0: \rho = 0$) and Coefficient of Determination ($R^2$).
5. **Main Model - Markowitz Modern Portfolio Theory (MPT)**: Analytical two-asset variance formula vs. observed series empirical variance mathematical equivalence proof ($|\sigma_p^2 - s_{p,\text{observed}}^2| < 10^{-6}$) and Minimum Variance Portfolio (MVP) optimization curve.

In [ ]:
# 1. Environment Setup & Fixed Random Seeds
import os
import sys
import random
import math
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt
import seaborn as sns

# Enforce fixed random seed for 100% deterministic reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Set dark styling consistent with application UI
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Reproducibility Seed Initialized: {RANDOM_SEED}")

## 2. Load Historical Market Data
We load historical daily prices exported directly from the database (`data/raw/historical_prices.csv`).

In [ ]:
df_prices = pd.read_csv('../data/raw/historical_prices.csv')
df_prices['date'] = pd.to_datetime(df_prices['date'])
df_prices = df_prices.sort_values(by=['symbol', 'date']).reset_index(drop=True)

print(f"Total Price Records: {len(df_prices)}")
print(f"Unique Equities: {df_prices['symbol'].nunique()}")
print(f"Date Range: {df_prices['date'].min().strftime('%Y-%m-%d')} to {df_prices['date'].max().strftime('%Y-%m-%d')}")
df_prices.head()

## 3. Compute Simple Daily Returns
$$\displaystyle R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$
Day 0 for each stock has no preceding price and is dropped (never zero-filled).

In [ ]:
# Pivot closing prices by date for all 10 stocks
price_matrix = df_prices.pivot(index='date', columns='symbol', values='adjusted_close').dropna()

# Calculate percentage returns (first day dropped)
returns_matrix = price_matrix.pct_change().dropna()

print(f"Aligned Synchronized Sessions (n): {len(returns_matrix)}")
print(f"Asset Universe: {list(returns_matrix.columns)}")
returns_matrix.head()

## 4. Comprehensive Descriptive Statistics & 95% Confidence Intervals

- **Unbiased Sample Variance (Bessel's Correction, $ddof=1$)**:
  $$\displaystyle s^2 = \frac{1}{n-1} \sum_{t=1}^n (R_t - \bar{R})^2$$
- **95% Confidence Interval for Mean** (Student's $t$ with $\nu = n-1$ degrees of freedom):
  $$\displaystyle \bar{R} \pm t_{0.975, n-1} \cdot \frac{s}{\sqrt{n}}$$
- **95% Confidence Interval for Variance** (Chi-Square $\chi^2$):
  $$\displaystyle \left[ \frac{(n-1)s^2}{\chi_{0.975, n-1}^2}, \frac{(n-1)s^2}{\chi_{0.025, n-1}^2} \right]$$

In [ ]:
desc_stats = []
alpha = 0.05

for sym in returns_matrix.columns:
    r = returns_matrix[sym].values
    n = len(r)
    mean_r = np.mean(r)
    median_r = np.median(r)
    trimmed_mean = stats.trim_mean(r, 0.05)
    var_sample = np.var(r, ddof=1)  # Bessel's correction
    s = np.std(r, ddof=1)
    iqr = stats.iqr(r)
    mad = stats.median_abs_deviation(r)
    ann_vol = s * np.sqrt(252)
    skew = stats.skew(r)
    kurt = stats.kurtosis(r, fisher=True)
    
    # 95% CI for Mean Return (t-distribution)
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    se_mean = s / np.sqrt(n)
    ci_mean_lower = mean_r - t_crit * se_mean
    ci_mean_upper = mean_r + t_crit * se_mean
    
    # 95% CI for Variance & Volatility (chi-square distribution)
    chi2_lower = stats.chi2.ppf(alpha / 2, df=n - 1)
    chi2_upper = stats.chi2.ppf(1 - alpha / 2, df=n - 1)
    ci_vol_lower = np.sqrt((n - 1) * var_sample / chi2_upper) * np.sqrt(252)
    ci_vol_upper = np.sqrt((n - 1) * var_sample / chi2_lower) * np.sqrt(252)
    
    desc_stats.append({
        'Symbol': sym,
        'n': n,
        'Daily Mean (%)': mean_r * 100,
        '95% CI Mean (%)': f"[{ci_mean_lower*100:.3f}%, {ci_mean_upper*100:.3f}%]",
        'Median (%)': median_r * 100,
        'Trimmed Mean (%)': trimmed_mean * 100,
        'Daily Vol (s %)': s * 100,
        'Ann. Vol (σ %)': ann_vol * 100,
        '95% CI Ann. Vol (%)': f"[{ci_vol_lower*100:.2f}%, {ci_vol_upper*100:.2f}%]",
        'Skewness': skew,
        'Excess Kurtosis': kurt
    })

df_desc = pd.DataFrame(desc_stats)
df_desc

## 5. Econometric Assumption Checks

### 5.1 Normality Diagnostics (Jarque-Bera & Shapiro-Wilk)
Testing whether returns conform to a Gaussian normal distribution ($H_0$: normality holds):

In [ ]:
normality_results = []
for sym in returns_matrix.columns:
    r = returns_matrix[sym].values
    jb_stat, jb_pval = stats.jarque_bera(r)
    sw_stat, sw_pval = stats.shapiro(r)
    normality_results.append({
        'Symbol': sym,
        'Jarque-Bera Stat': jb_stat,
        'JB p-value': jb_pval,
        'JB Decision (α=0.05)': 'Reject (Non-normal)' if jb_pval < 0.05 else 'Fail to Reject',
        'Shapiro-Wilk W': sw_stat,
        'SW p-value': sw_pval,
        'SW Decision (α=0.05)': 'Reject (Non-normal)' if sw_pval < 0.05 else 'Fail to Reject'
    })

df_norm = pd.DataFrame(normality_results)
df_norm

### 5.2 Quantile-Quantile (Q-Q) Plots
Empirical quantiles plotted against standard normal theoretical quantiles illustrate fat tails (leptokurtosis) and tail risk deviations.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for idx, sym in enumerate(returns_matrix.columns):
    r = returns_matrix[sym].values
    stats.probplot(r, dist='norm', plot=axes[idx])
    axes[idx].set_title(f'Q-Q Plot: {sym}', fontsize=11, fontweight='bold')
    axes[idx].get_lines()[0].set_markerfacecolor('#3B82F6')
    axes[idx].get_lines()[0].set_markeredgecolor('#1D4ED8')
    axes[idx].get_lines()[0].set_markersize(4.0)
    axes[idx].get_lines()[1].set_color('#EF4444')
    axes[idx].get_lines()[1].set_linewidth(1.8)
plt.tight_layout()
plt.show()

### 5.3 Timeseries Stationarity (Augmented Dickey-Fuller Test)
Testing for unit roots ($H_0$: unit root non-stationary vs. $H_1$: stationary $I(0)$):

In [ ]:
stationarity_results = []
for sym in returns_matrix.columns:
    p = price_matrix[sym].values
    r = returns_matrix[sym].values
    adf_p = adfuller(p, autolag='AIC', result_object=False)
    adf_r = adfuller(r, autolag='AIC', result_object=False)
    stationarity_results.append({
        'Symbol': sym,
        'Price ADF Stat': adf_p[0],
        'Price p-value': adf_p[1],
        'Price Stationarity': 'Non-Stationary I(1)' if adf_p[1] >= 0.05 else 'Stationary',
        'Return ADF Stat': adf_r[0],
        'Return p-value': adf_r[1],
        'Return Stationarity': 'Stationary I(0)' if adf_r[1] < 0.05 else 'Non-Stationary'
    })

df_stat = pd.DataFrame(stationarity_results)
df_stat

### 5.4 Serial Independence & Autocorrelation Checks
- **Durbin-Watson ($d \approx 2.0$)**: First-order serial independence.
- **Ljung-Box**: Cumulative serial correlation across lags 1, 5, 10.

In [ ]:
serial_results = []
for sym in returns_matrix.columns:
    r = returns_matrix[sym].values
    dw = durbin_watson(r - np.mean(r))
    lb = acorr_ljungbox(r, lags=[1, 5, 10], return_df=True)
    serial_results.append({
        'Symbol': sym,
        'Durbin-Watson Stat': dw,
        'DW Interpretation': 'Zero Autocorrelation (Random Walk)' if 1.8 <= dw <= 2.2 else 'Autocorrelated',
        'Ljung-Box Lag 1 p-val': lb.loc[1, 'lb_pvalue'],
        'Ljung-Box Lag 5 p-val': lb.loc[5, 'lb_pvalue'],
        'Ljung-Box Lag 10 p-val': lb.loc[10, 'lb_pvalue']
    })

df_serial = pd.DataFrame(serial_results)
df_serial

### 5.5 Heteroskedasticity & Volatility Clustering (ARCH Effect)
Testing whether large shocks follow large shocks by evaluating the autocorrelation of demeaned squared returns $(R_t - \bar{R})^2$.

In [ ]:
arch_results = []
for sym in returns_matrix.columns:
    r = returns_matrix[sym].values
    demeaned_sq = (r - np.mean(r)) ** 2
    sq_autocorr = float(np.corrcoef(demeaned_sq[:-1], demeaned_sq[1:])[0, 1])
    has_clustering = sq_autocorr > 0.05
    arch_results.append({
        'Symbol': sym,
        'Squared Returns Lag-1 Autocorr': sq_autocorr,
        'Volatility Clustering Detected?': 'YES' if has_clustering else 'NO',
        'ARCH Implication': 'Mandelbrot clustering (large shocks cluster)' if has_clustering else 'Constant conditional variance'
    })

df_arch = pd.DataFrame(arch_results)
df_arch

## 6. Hypothesis Testing & Standardized Effect Sizes

### 6.1 One-Sample $t$-Test on Mean Returns ($H_0: \mu = 0$)
- **Cohen's $d$ Effect Size**: $\displaystyle d = \frac{\bar{R} - \mu_0}{s}$
- **Annualized Sharpe Ratio**: $\displaystyle \text{SR} = \frac{\text{Annualized Return} - R_f}{\text{Annualized Volatility}}$, with benchmark risk-free rate $R_f = 6.5\%$ (RBI 91-day T-Bill rate).

In [ ]:
tests_and_effects = []
risk_free_rate = 0.065

for sym in returns_matrix.columns:
    r = returns_matrix[sym].values
    t_res = stats.ttest_1samp(r, popmean=0.0)
    wilc = stats.wilcoxon(r, alternative='two-sided')
    s = np.std(r, ddof=1)
    mean_r = np.mean(r)
    cohens_d = mean_r / s if s > 0 else 0.0
    
    ann_ret = mean_r * 252
    ann_vol = s * np.sqrt(252)
    sharpe = (ann_ret - risk_free_rate) / ann_vol if ann_vol > 0 else 0.0
    
    tests_and_effects.append({
        'Symbol': sym,
        'Daily Mean (%)': mean_r * 100,
        't-statistic': t_res.statistic,
        't-test p-value': t_res.pvalue,
        'Decision (α=0.05)': 'Reject (μ != 0)' if t_res.pvalue < 0.05 else 'Fail to Reject',
        'Wilcoxon p-val': wilc.pvalue,
        "Cohen's d": cohens_d,
        'Effect Magnitude': 'Negligible (|d|<0.2)' if abs(cohens_d) < 0.2 else 'Small (0.2<=|d|<0.5)',
        'Annualized Sharpe (Rf=6.5%)': sharpe
    })

df_tests = pd.DataFrame(tests_and_effects)
df_tests

### 6.2 Pairwise Pearson Correlation Tests & Fisher $z$ Confidence Intervals
- **Exact Student's $t$ Significance Test** ($H_0: \rho = 0$):
  $$\displaystyle t = r \sqrt{\frac{n-2}{1 - r^2}} \sim t_{n-2}$$
- **Fisher $z$-Transformation 95% Confidence Interval**:
  $$\displaystyle z = \frac{1}{2} \ln \left( \frac{1 + r}{1 - r} \right) = \text{arctanh}(r), \quad \text{SE}_z = \frac{1}{\sqrt{n-3}}$$
- **Coefficient of Determination ($R^2 = r^2$)**: Proportion of shared variance between assets.

In [ ]:
pairwise_results = []
cols = list(returns_matrix.columns)
n = len(returns_matrix)

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        s1 = cols[i]
        s2 = cols[j]
        r1 = returns_matrix[s1].values
        r2 = returns_matrix[s2].values
        r_val, p_val = stats.pearsonr(r1, r2)
        
        # Fisher z 95% CI
        z = np.arctanh(r_val)
        se_z = 1.0 / np.sqrt(n - 3)
        z_crit = stats.norm.ppf(0.975)
        ci_lower = np.tanh(z - z_crit * se_z)
        ci_upper = np.tanh(z + z_crit * se_z)
        r_sq = r_val ** 2
        
        pairwise_results.append({
            'Pair': f"{s1} & {s2}",
            'Pearson r': r_val,
            'p-value': p_val,
            '95% CI for r': f"[{ci_lower:.3f}, {ci_upper:.3f}]",
            'R² (Shared Var)': r_sq,
            'Effect Size': 'Large (|r|>=0.5)' if abs(r_val)>=0.5 else ('Medium (0.3<=|r|<0.5)' if abs(r_val)>=0.3 else 'Small/Negligible')
        })

df_pairs = pd.DataFrame(pairwise_results).sort_values(by='Pearson r', ascending=False).reset_index(drop=True)
print(f"Total Unique Cross-Asset Pairs (10 C 2): {len(df_pairs)}")
df_pairs.head(10)

## 7. Diverging Pearson Correlation Heatmap
Visualizing the complete 10×10 correlation structure with divergent palette centered at zero ($0.00$):

In [ ]:
corr_matrix = returns_matrix.corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="vlag", 
    center=0.0, 
    vmin=-1.0, 
    vmax=1.0, 
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation (r)'}
)
plt.title('10x10 Pearson Correlation Heatmap (Approved NSE Equities)', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 8. Main Model - Markowitz Modern Portfolio Theory (MPT) Verification

### 8.1 Analytical Variance Formula vs. Observed Series Sample Variance
$$\displaystyle \sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2w_1w_2 \text{Cov}(R_1, R_2)$$
We test this across 101 allocation weights from $w_1 = 0.0$ to $w_1 = 1.0$ against the empirical sample variance ($ddof=1$) of the linear returns series $R_{p,t} = w_1 R_{1,t} + w_2 R_{2,t}$:

In [ ]:
sym1 = 'SOLARINDS.NS'
sym2 = 'BAJFINANCE.NS'

r1 = returns_matrix[sym1]
r2 = returns_matrix[sym2]

cov_val = r1.cov(r2)  # ddof=1
var1 = r1.var(ddof=1)
var2 = r2.var(ddof=1)

weights = np.linspace(0.0, 1.0, 101)
port_vols = []
port_returns = []
max_discrepancy = 0.0

for w1 in weights:
    w2 = 1.0 - w1
    # Analytical Formula Variance
    var_theoretical = (w1**2 * var1) + (w2**2 * var2) + (2 * w1 * w2 * cov_val)
    vol_analytical = np.sqrt(var_theoretical) * np.sqrt(252)
    
    # Observed Return Series Sample Variance
    r_portfolio = (w1 * r1) + (w2 * r2)
    var_observed = r_portfolio.var(ddof=1)
    
    # Discrepancy
    diff = abs(var_theoretical - var_observed)
    if diff > max_discrepancy:
        max_discrepancy = diff
        
    ret_portfolio = ((w1 * r1.mean()) + (w2 * r2.mean())) * 252
    port_vols.append(vol_analytical * 100)
    port_returns.append(ret_portfolio * 100)

print(f"Maximum Absolute Discrepancy between Theoretical Formula & Observed Variance: {max_discrepancy:.2e}")
assert max_discrepancy < 1e-6, "Proof Failed!"
print("✓ Mathematical Equivalence Proved: Delta < 10^-6 across all weight allocations!")

# Plot Two-Stock Portfolio Risk-Return Curve
plt.figure(figsize=(10, 6))
plt.plot(port_vols, port_returns, color='#2563eb', linewidth=2.5, label='Two-Asset Investment Opportunity Set')
plt.scatter([port_vols[0], port_vols[-1]], [port_returns[0], port_returns[-1]], color=['#dc2626', '#16a34a'], s=80, zorder=5)
plt.annotate(f'100% {sym2}', (port_vols[0], port_returns[0]), textcoords="offset points", xytext=(10,-5))
plt.annotate(f'100% {sym1}', (port_vols[-1], port_returns[-1]), textcoords="offset points", xytext=(10,-5))

# Mark Minimum Variance Portfolio (MVP)
min_idx = np.argmin(port_vols)
plt.scatter(port_vols[min_idx], port_returns[min_idx], color='#d97706', s=140, marker='*', zorder=6, label=f'Minimum Variance Portfolio (w1={weights[min_idx]:.2f}, Vol={port_vols[min_idx]:.1f}%)')

plt.title(f'Markowitz Portfolio Frontier: {sym1} & {sym2}', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Annualized Volatility (%)', fontsize=11)
plt.ylabel('Annualized Return (%)', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()